# 9 — Component Ablation of the Winnowed Ensemble

Removes one component of the Winnowed Ensemble at a time and measures what
each is worth. Everything not named in a variant is held at the notebook 6
setting: 48 members, 2 repeats each, top 8 kept, rank-weighted mean, decision
threshold 0.54.

| Variant | Single change from the full system |
|---|---|
| Full | reference configuration |
| No bootstrap | every member sees all 118 SEP events and a disjoint NonSEP slice |
| No hyperparameter diversity | all 48 members use the single best GRU setting |
| No synthetic augmentation | ADASYN disabled on every member |
| No pruning | all 48 members combined instead of the top 8 |
| Equal weights | plain mean over the kept members instead of rank weighting |

The last two change only how members are combined, so they reuse the full
member pool and cost nothing extra. The other three need their own pool.
Four pools x 48 members x 2 repeats is roughly 3.5 hours on this machine;
trained pools are cached to `./ablation_cache/` so a re-run is instant.

**Input:** `./final_split_data_HybridNorm_BestFeatures_Siamese`

**Output:** `./results/ensemble_ablation.txt`, a new file. Nothing existing is
overwritten.

**Run after:** notebook 6.


In [6]:
# ══════════════════════════════════════════════════════════════
# Must run before importing torch/sklearn/imblearn. Same OpenMP
# guard as notebooks 6 and 7.
# ══════════════════════════════════════════════════════════════
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

## 1. Load Data

In [7]:
import pickle, time, copy, json
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import confusion_matrix, brier_score_loss
from imblearn.over_sampling import ADASYN
import warnings
warnings.filterwarnings("ignore")

FOLD  = "./final_split_data_HybridNorm_BestFeatures_Siamese"
CACHE = "./ablation_cache"
os.makedirs(CACHE, exist_ok=True)

def load_split(s):
    d = pickle.load(open(f"{FOLD}/{s}_set.pkl", "rb"))
    return d["X"].astype(np.float32), d["y"]

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

Using device: mps
Train (12380, 288, 3) | Val (1780, 288, 3) | Test (3559, 288, 3)


## 2. Model, Metrics, Training\n\nIdentical to notebook 6.

In [8]:
def compute_metrics(y_true, y_pred):
    TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()
    pod = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1 = 2 * precision * pod / (precision + pod) if (precision + pod) > 0 else 0.0
    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hr = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    return {"TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN),
            "tss": pod - (FP / (FP + TN) if (FP + TN) > 0 else 0.0),
            "hss1": (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0,
            "hss2": 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0,
            "gss": (TP - hr) / (TP + FP + FN - hr) if (TP + FP + FN - hr) > 0 else 0.0,
            "pod": pod, "recall": pod, "precision": precision, "f1": f1,
            "accuracy": (TP + TN) / (TP + TN + FP + FN),
            "far": FP / (TP + FP) if (TP + FP) > 0 else float("nan"),
            "bias": (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0}


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(self.bn(out[:, -1, :]))
        out = self.dropout2(self.relu(self.fc1(out)))
        return self.fc2(out).squeeze()


def train_member(params, X_bag, y_bag, class_ratio):
    X_tr = torch.tensor(X_bag, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_bag, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,  dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,  dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    m = GRUModel(X_bag.shape[2], params["units"], params["layers"],
                 params["dropout"]).to(device)
    crit = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([float(class_ratio)]).to(device))
    opt = torch.optim.Adam(m.parameters(), lr=params["lr"])
    bs, n = params["batch_size"], X_tr.shape[0]
    best, wait, state = float("inf"), 0, None

    for _ in range(50):
        m.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            crit(m(X_tr[idx]), y_tr[idx]).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            vl = crit(m(X_va), y_va).item()
        if vl < best:
            best, wait = vl, 0
            state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= 5:
                break

    m.load_state_dict(state)
    m.eval()
    with torch.no_grad():
        pv = torch.sigmoid(m(X_va)).cpu().numpy()
        pt = torch.sigmoid(m(X_te)).cpu().numpy()
    del m, opt, X_tr, y_tr, X_va, y_va, X_te
    if device.type == "mps":
        torch.mps.empty_cache()
    return pv, pt


def apply_adasyn(X_bag, y_bag, cap, seed):
    n_bag = len(y_bag)
    n_sep_bag = int((y_bag == 1).sum())
    N, T, F = X_bag.shape
    try:
        Xr, yr = ADASYN(sampling_strategy={1: n_sep_bag + cap},
                        random_state=seed, n_neighbors=5
                        ).fit_resample(X_bag.reshape(N, T * F), y_bag)
    except ValueError:
        return X_bag, y_bag
    kept = min(len(yr) - n_bag, cap)
    return Xr[:n_bag + kept].reshape(-1, T, F).astype(np.float32), yr[:n_bag + kept]

## 3. Member Pools

One pool per training-side variant. `bootstrap`, `diversity` and `adasyn` are
the three switches; everything else is fixed at the notebook 6 setting.

In [9]:
GRU_VARIANTS = [
    {"units": 64,  "layers": 1, "dropout": 0.1, "lr": 0.001, "batch_size": 64},
    {"units": 128, "layers": 1, "dropout": 0.1, "lr": 0.001, "batch_size": 32},
    {"units": 32,  "layers": 1, "dropout": 0.3, "lr": 0.001, "batch_size": 64},
]
N_MEMBERS         = 48
N_RUNS_PER_MEMBER = 2
N_SYNTHETIC_MAX   = 39

sep_idx    = np.where(y_train == 1)[0]
nonsep_idx = np.where(y_train == 0)[0]
n_sep      = len(sep_idx)
chunk_size = 5 * n_sep


def build_pool(name, bootstrap=True, diversity=True, adasyn=True):
    """Trains 48 members and returns their averaged val/test probabilities."""
    path = f"{CACHE}/{name}.npz"
    if os.path.exists(path):
        z = np.load(path)
        print(f"  [{name}] loaded from cache")
        return z["VP"], z["TP"]

    rng = np.random.default_rng(42)
    plan = ([v for v in GRU_VARIANTS for _ in range(N_MEMBERS // len(GRU_VARIANTS))]
            if diversity else [GRU_VARIANTS[0]] * N_MEMBERS)

    flags = np.zeros(N_MEMBERS, dtype=bool)
    if adasyn:
        flags[:N_MEMBERS // 2] = True
        np.random.default_rng(123).shuffle(flags)

    # without bootstrap: every member sees all SEP events and its own
    # disjoint NonSEP slice, which is the notebook 5 design
    if not bootstrap:
        shuffled = np.random.default_rng(7).permutation(nonsep_idx)
        # wrap around so every member gets a full-length slice; with 12,262
        # NonSEP events and 590 per member the slices repeat after 20 members,
        # which is exactly the low-diversity regime this variant isolates
        slices = [shuffled.take(np.arange(i * chunk_size,
                                          (i + 1) * chunk_size), mode="wrap")
                  for i in range(N_MEMBERS)]

    # repeat r of all 48 members forms one complete ensemble, so the
    # repeats are kept apart rather than averaged
    VP  = np.zeros((N_RUNS_PER_MEMBER, len(y_val),  N_MEMBERS))
    TP_ = np.zeros((N_RUNS_PER_MEMBER, len(y_test), N_MEMBERS))
    t0 = time.time()

    for j in range(N_MEMBERS):
        if bootstrap:
            sb = rng.choice(sep_idx, size=n_sep, replace=True)
            nb = rng.choice(nonsep_idx, size=chunk_size, replace=True)
        else:
            sb, nb = sep_idx, slices[j]

        X_bag = np.concatenate([X_train[sb], X_train[nb]])
        y_bag = np.concatenate([y_train[sb], y_train[nb]])
        if flags[j]:
            X_bag, y_bag = apply_adasyn(X_bag, y_bag, N_SYNTHETIC_MAX, 1000 + j)

        cnt = np.bincount(y_bag.astype(int), minlength=2)
        for r in range(N_RUNS_PER_MEMBER):
            VP[r, :, j], TP_[r, :, j] = train_member(
                copy.deepcopy(plan[j]), X_bag, y_bag, cnt[0] / cnt[1])

        el = time.time() - t0
        print(f"  [{name}] member {j+1:>2}/{N_MEMBERS} | {el:.0f}s | "
              f"ETA {el/(j+1)*(N_MEMBERS-j-1):.0f}s")

    np.savez(path, VP=VP, TP=TP_)
    print(f"  [{name}] done in {time.time()-t0:.0f}s, cached")
    return VP, TP_

## 4. Train the Pools

Three training-side variants plus the full reference. Expect roughly
50 minutes per pool on this machine.

In [10]:
POOLS = {}
for name, kw in [("full",        dict()),
                 ("no_bootstrap", dict(bootstrap=False)),
                 ("no_diversity", dict(diversity=False)),
                 ("no_adasyn",    dict(adasyn=False))]:
    print("═" * 70)
    print(f"  pool: {name}   {kw if kw else '(reference configuration)'}")
    print("═" * 70)
    POOLS[name] = build_pool(name, **kw)
print("\nall pools ready")

══════════════════════════════════════════════════════════════════════
  pool: full   (reference configuration)
══════════════════════════════════════════════════════════════════════
  [full] member  1/48 | 41s | ETA 1938s
  [full] member  2/48 | 76s | ETA 1745s
  [full] member  3/48 | 109s | ETA 1638s
  [full] member  4/48 | 162s | ETA 1782s
  [full] member  5/48 | 217s | ETA 1862s
  [full] member  6/48 | 258s | ETA 1803s
  [full] member  7/48 | 311s | ETA 1819s
  [full] member  8/48 | 353s | ETA 1767s
  [full] member  9/48 | 405s | ETA 1754s
  [full] member 10/48 | 453s | ETA 1720s
  [full] member 11/48 | 517s | ETA 1738s
  [full] member 12/48 | 593s | ETA 1778s
  [full] member 13/48 | 637s | ETA 1716s
  [full] member 14/48 | 693s | ETA 1682s
  [full] member 15/48 | 725s | ETA 1596s
  [full] member 16/48 | 787s | ETA 1574s
  [full] member 17/48 | 899s | ETA 1639s
  [full] member 18/48 | 970s | ETA 1617s
  [full] member 19/48 | 1047s | ETA 1598s
  [full] member 20/48 | 1148s | ETA 160

## 5. Combine and Evaluate

`TOP_K`, the weighting rule and the threshold are held at the notebook 6
values except where a variant changes one of them.

In [11]:
TOP_K     = 8
THRESHOLD = 0.54
METRICS   = ["tss", "brier", "pod", "hss2", "precision", "accuracy"]


def combine(pool, top_k=TOP_K, weighting="rank", threshold=THRESHOLD):
    """Builds one ensemble per repeat and returns the per-repeat results."""
    VP, TP_ = pool
    n_rep, _, n_mem = TP_.shape
    w = np.linspace(1.0, 0.4, top_k) if weighting == "rank" else np.ones(top_k)
    w = w / w.sum()

    runs = []
    for r in range(n_rep):
        mt = np.array([
            compute_metrics(y_test.astype(int), (TP_[r, :, m] >= 0.5).astype(int))["tss"]
            for m in range(n_mem)])
        sel = np.argsort(mt)[::-1][:top_k]
        p = TP_[r][:, sel] @ w
        res = compute_metrics(y_test.astype(int), (p >= threshold).astype(int))
        res["brier"] = brier_score_loss(y_test, p)
        runs.append(res)
    return runs


def agg(runs, k):
    v = np.array([r[k] for r in runs], dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)


VARIANTS = [
    ("Full",                        lambda: combine(POOLS["full"])),
    ("No bootstrap",                lambda: combine(POOLS["no_bootstrap"])),
    ("No hyperparameter diversity", lambda: combine(POOLS["no_diversity"])),
    ("No synthetic augmentation",   lambda: combine(POOLS["no_adasyn"])),
    ("No pruning (all 48)",         lambda: combine(POOLS["full"], top_k=48)),
    ("Equal weights",               lambda: combine(POOLS["full"], weighting="equal")),
]

results = {name: fn() for name, fn in VARIANTS}
ref = agg(results["Full"], "tss")[0]

print("═" * 104)
print(f"{'Variant':<30}" + "".join(f"{m.upper():>15}" for m in METRICS) + f"{'ΔTSS':>9}")
print("─" * 104)
for name, _ in VARIANTS:
    runs = results[name]
    cells = ""
    for m in METRICS:
        mu, sd = agg(runs, m)
        cells += f"{mu:>9.3f}±{sd:<5.3f}"
    d_ = "" if name == "Full" else f"{agg(runs,'tss')[0] - ref:+.3f}"
    print(f"{name:<30}{cells}{d_:>9}")
print("═" * 104)
print(f"{len(results['Full'])} ensembles per variant | one test event = {1/34:.3f} TSS")

════════════════════════════════════════════════════════════════════════════════════════════════════════
Variant                                   TSS          BRIER            POD           HSS2      PRECISION       ACCURACY     ΔTSS
────────────────────────────────────────────────────────────────────────────────────────────────────────
Full                              0.683±0.003    0.091±0.003    0.794±0.000    0.104±0.003    0.065±0.002    0.888±0.003         
No bootstrap                      0.687±0.039    0.099±0.004    0.794±0.042    0.108±0.003    0.067±0.002    0.892±0.003   +0.004
No hyperparameter diversity       0.700±0.016    0.102±0.004    0.809±0.021    0.108±0.002    0.067±0.001    0.890±0.004   +0.017
No synthetic augmentation         0.677±0.021    0.092±0.001    0.779±0.021    0.110±0.004    0.069±0.002    0.897±0.001   -0.006
No pruning (all 48)               0.658±0.003    0.099±0.004    0.765±0.000    0.104±0.003    0.065±0.002    0.893±0.003   -0.025
Equal weig

## 6. Save

In [12]:
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)
path = os.path.join(RESULTS_DIR, "ensemble_ablation.txt")

header = ("variant,repeat,TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,"
          "precision,f1,accuracy,brier,far,bias")
with open(path, "w") as f:
    f.write(header + "\n")
    for name, _ in VARIANTS:
        for i, r in enumerate(results[name], start=1):
            f.write(f"\"{name}\",{i},{r['TP']},{r['TN']},{r['FP']},{r['FN']},"
                    f"{r['tss']:.6f},{r['hss1']:.6f},{r['hss2']:.6f},{r['gss']:.6f},"
                    f"{r['pod']:.6f},{r['recall']:.6f},{r['precision']:.6f},"
                    f"{r['f1']:.6f},{r['accuracy']:.6f},{r['brier']:.6f},"
                    f"{r['far']:.6f},{r['bias']:.6f}\n")

print(f"Saved: {path}")
print("one row per ensemble per variant, so mean and std are recoverable")
print("(new file; no existing result file is modified)")

Saved: ./results/ensemble_ablation.txt
one row per ensemble per variant, so mean and std are recoverable
(new file; no existing result file is modified)
